# Credit Card Fraud Detection - Analysis Notebook

**Student 1:** Marco Dal Mas 886663
**Student 2:** Rares Stefan Neagu 906503
**Student 3:** Matilde Dovizio 904660

## Project Overview

This project focuses on the detection of fraudulent credit card transactions using a real-world transaction dataset. Each transaction is labeled either as legitimate or fraudulent, and the main objective is to build a classification model able to identify suspicious transactions.

The project track already indicates that the dataset is highly imbalanced, meaning that fraudulent transactions are much rarer than legitimate ones. This is a central aspect of the problem: in fraud detection, a model that simply predicts every transaction as legitimate could still obtain a very high accuracy, but it would be useless from a business perspective because it would fail to detect the fraud cases. For this reason, the notebook does not rely only on accuracy, but evaluates the models using more appropriate metrics such as precision, recall, F1-score, ROC-AUC, and especially PR-AUC.

The analysis follows a practical data science pipeline. We first load the dataset and inspect its structure, then we analyze the target variable to understand the actual level of imbalance. After that, we check data quality, explore the main numerical distributions, prepare the features correctly, and compare multiple classification models. Particular attention is paid to avoiding data leakage, meaning that all preprocessing steps must be fitted only on the training data and not on the test set.

From a business point of view, the two types of classification errors do not have the same cost. A false positive means that a legitimate transaction is incorrectly flagged as fraudulent. This may annoy the customer, block a valid payment, and create operational costs. A false negative means that a real fraudulent transaction is classified as legitimate. This is usually more dangerous, because it can generate direct financial losses and security risks. For this reason, in this project we prefer a slightly more conservative approach: it is generally better to accept some additional false alarms if this helps the model detect more real frauds.

The notebook compares three classification models with different levels of complexity. Logistic Regression is used as a simple and interpretable baseline model, because it estimates the probability that a transaction belongs to the fraud class. Random Forest is used as a more flexible ensemble model based on many decision trees, where the final prediction is obtained through majority voting. Finally, XGBoost is used as a more advanced boosting model, which builds trees sequentially and tries to correct the errors made by previous trees.

For each model, we first evaluate a default version and then apply hyperparameter tuning to check whether performance can be improved. The final goal is not only to find the model with the best numerical score, but also to understand which model offers the best trade-off between detecting fraudulent transactions and limiting unnecessary false alarms.


## 1. Setup

We start with the technical setup of the notebook. All the required libraries are imported at the beginning, even if some of them are used only later in the analysis. This makes the notebook easier to run and check, because possible missing dependencies or import errors are detected immediately, before executing longer and more expensive operations such as model training or hyperparameter tuning.

The random seed is also fixed in this section. Some steps of the project, such as the train/test split, model training, and randomized hyperparameter search, include random operations. Setting a fixed seed makes the results reproducible, meaning that the same code can be run again and produce consistent results.

This is particularly useful in a group project, because all members need to obtain the same results in order to discuss the models and write consistent conclusions. We also tested the notebook without fixing the random seed and verified that the results remain stable, but keeping the seed fixed makes the workflow safer, clearer, and easier to reproduce.

In [ ]:
# Utilities for file and folder path handling
from pathlib import Path

# Used to suppress non-critical warning messages
import warnings

# Numerical computations and array manipulation
import numpy as np

# Data loading, cleaning, and analysis
import pandas as pd

# Interactive data visualizations
import plotly.express as px

# Display pandas DataFrames nicely in Jupyter notebooks
from IPython.display import display

# Specific warning generated when machine learning models fail to converge
from sklearn.exceptions import ConvergenceWarning

# Tools for model evaluation, validation, and dataset splitting
from sklearn.model_selection import (
    GridSearchCV,          # Exhaustive hyperparameter search
    RandomizedSearchCV,    # Random hyperparameter search
    StratifiedKFold,       # Stratified cross-validation
    train_test_split,      # Split data into training and testing sets
)

# Robust feature scaling, less sensitive to outliers
from sklearn.preprocessing import RobustScaler

# Logistic Regression classification model
from sklearn.linear_model import LogisticRegression

# Random Forest classification model
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import confusion_matrix

# Performance evaluation metrics
from sklearn.metrics import (
    auc,                      # Generic area under a curve
    average_precision_score,  # Area under Precision-Recall curve
    f1_score,                 # Harmonic mean of precision and recall
    precision_recall_curve,   # Points for Precision-Recall curve
    precision_score,          # Precision metric
    recall_score,             # Recall metric
    roc_auc_score,            # Area under ROC curve
    roc_curve,                # Points for ROC curve
)

# Lower-level Plotly API used to build custom interactive figures
import plotly.graph_objects as go

# Import XGBoost classifier if available
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is required for the XGBoost modelling section. Install it with: pip install xgboost"
    ) from exc

# Fixed random seed for reproducibility
RANDOM_STATE = 42

DATA_PATH = Path("creditcard.csv")

# Display settings for pandas DataFrames
pd.set_option("display.max_columns", 40)  
pd.set_option("display.float_format", lambda x: f"{x:.6f}")  

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Loading the data

The first practical step is simply to load the file and make sure the notebook can find it.

At this point we only know that the project track is about credit-card fraud detection and that the dataset should contain a target column. We still need to inspect the table before deciding how to prepare and evaluate the models.


In [ ]:
# Check whether the dataset file exists in the specified location
if not DATA_PATH.exists():
    # Raise an error if the file cannot be found
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Place creditcard.csv in the same folder as this notebook."
    )

# Load the credit card transactions dataset into a pandas DataFrame
df = pd.read_csv(DATA_PATH)

# Confirm that the dataset has been successfully loaded
print(f"Loaded dataset: {DATA_PATH}")

## 3. First Inspection of the Dataset

Before building any model, we ask a few basic questions about the dataset:

* How many rows and columns are there?
* Which columns are available?
* Is the target column really present?
* Which values does the target column contain?
* Are there missing values?
* Is one class much rarer than the other?
* Which variables are directly interpretable?

This is the first checkpoint of the analysis. As discussed in class, the performance of a machine learning model depends strongly on the quality and structure of the data. A good algorithm cannot compensate for missing, inconsistent, badly represented, or misunderstood data. For this reason, before preparing the features and training the models, we first inspect the dataset to understand what we are working with.

After looking at the available columns, we can also identify `Time` and `Amount` as the only directly interpretable numerical variables. The other features are anonymized transformed variables, so they can still be useful for modelling, but they are less meaningful for direct interpretation.


In [ ]:
print("Dataset shape (rows, cols):", df.shape)

print("Number of columns:", df.shape[1])

print("\nColumns:")
print(df.columns.to_list())

assert "Class" in df.columns, "The target column 'Class' is missing."

observed_classes = sorted(df["Class"].dropna().unique().tolist())
print("\nObserved values in target column 'Class':", observed_classes)

# Ensure that the classification problem is binary (0 = legitimate, 1 = fraud)
assert observed_classes == [0, 1], "Expected a binary target encoded as 0 and 1."

# Define the interpretation of the target labels
# 0 = legitimate transaction, 1 = fraudulent transaction
CLASS_LABELS = {
    0: "Legitimate (0)",
    1: "Fraud (1)",
}
print("Working label interpretation:", CLASS_LABELS)

display(df.head())

null_counts = df.isna().sum()

null_total = int(null_counts.sum())
print("\nTotal null values in dataset:", null_total)

if null_total == 0:
    print("No missing values detected.")
else:
    display(null_counts[null_counts > 0].sort_values(ascending=False))

class_counts = df["Class"].value_counts().sort_index()

class_rates = class_counts / len(df)

print("\nClass distribution (counts):")
print(class_counts)

print("\nClass distribution (rates):")
print((class_rates * 100).map(lambda x: f"{x:.6f}%"))

fraud_rate = float(class_rates.loc[1])

# Calculate the baseline accuracy obtained by always predicting the majority class
baseline_accuracy = float(class_rates.loc[0])

print(f"\nFraud prevalence: {fraud_rate:.6%}")
print(f"Majority-class baseline accuracy (always legitimate): {baseline_accuracy:.6%}")

# Generate descriptive statistics for the Time and Amount features
for col in ["Time", "Amount"]:
    print(f"\n{col} summary:")
    display(df[col].describe())

### What We Learn from the First Inspection

The dataset is loaded correctly. It contains **284,807 transactions** and **31 columns**.

The target column `Class` is present and contains only two observed values: `0` and `1`. Following the dataset convention, we interpret `0` as a legitimate transaction and `1` as a fraudulent transaction. This confirms that the task is a binary classification problem.

The class distribution confirms the main challenge of the project: fraudulent transactions are extremely rare. There are only **492 fraud transactions**, corresponding to about **0.173%** of the dataset. This means that a naive model that always predicts the majority class, namely “legitimate transaction”, would already obtain about **99.83% accuracy**, while detecting **zero frauds**. For this reason, accuracy alone is not a reliable metric for this project.

There are no missing values in the dataset, so no imputation is required. This simplifies the data preparation step, because we do not need to decide how to fill or remove incomplete observations.

The variables `Time` and `Amount` are the only directly interpretable numerical features in the dataset. `Amount` is strongly skewed: the median transaction amount is **22.00**, while the maximum value is **25,691.16**. This suggests the presence of extreme values and indicates that scaling should be handled carefully, especially for models that are sensitive to feature magnitude.


## 4. Exploratory Data Analysis

Now that the dataset has been loaded and its basic structure has been checked, we move to a more detailed exploratory analysis.

In this section, we use visualizations and summary statistics to answer a few specific questions:

1. **How severe is the class imbalance visually?**
2. **Does `Amount` behave differently for legitimate and fraudulent transactions?**
3. **Are some anonymized variables more related to the target than others?**
4. **Are there data-quality details, such as duplicated rows or zero-amount transactions, that we should know before modelling?**

The aim is not to force a strong interpretation from anonymized variables, because most features do not have a direct business meaning. Instead, the goal is to better understand the structure of the dataset and make more informed modelling choices.


In [ ]:
# Compute class distribution (counts) from the target variable
class_plot = (
    df["Class"]
    .value_counts()
    .rename_axis("Class")  
    .reset_index(name="Count")  
)

# Map numeric class labels to human-readable labels
class_plot["Label"] = class_plot["Class"].map(CLASS_LABELS)

# Create an interactive bar chart showing class imbalance
fig = px.bar(
    class_plot,
    x="Label",  
    y="Count",  
    text="Count",  
    title="Class distribution: legitimate transactions vs fraud",
)

# Improve bar text formatting and hover information
fig.update_traces(
    texttemplate="%{text:,}",  
    textposition="outside",    
    hovertemplate="%{x}<br>Count = %{y:,}<extra></extra>",
)

# Set axis labels and remove legend (not needed for single series)
fig.update_layout(
    xaxis_title="Class",
    yaxis_title="Count",
    showlegend=False
)

fig.show()

### Answer from the Class Distribution Plot

The plot confirms visually the strong imbalance already observed in the numerical counts. Legitimate transactions dominate the dataset, while fraudulent transactions are almost invisible on the same scale. This confirms that we are dealing with a rare-event classification problem.

This has an important consequence for model evaluation. A model could obtain a very high accuracy simply by predicting almost every transaction as legitimate, but this would not be useful for fraud detection. The positive class, fraud, is exactly the class we care about most.

For this reason, the following sections will not rely only on accuracy. We will focus on metrics that better describe the model’s ability to detect fraud:

* **Recall**, because a false negative means that a real fraud is missed;
* **Precision**, because too many false positives would create unnecessary checks and customer friction;
* **F1-score**, because it summarizes the balance between precision and recall at a selected threshold;
* **PR-AUC**, because it is especially informative when the positive class is very rare.

ROC-AUC will still be reported, because it measures the general ability of the model to separate the two classes. However, for the final comparison, PR-AUC is particularly important because it focuses more directly on the precision-recall trade-off for the fraud class.


In [ ]:
# ECDF (Empirical Cumulative Distribution Function) of transaction Amount grouped by class.
# A logarithmic x-axis is used because the Amount feature is highly right-skewed.
# Zero values are excluded only for visualization purposes since log(0) is undefined.
# They are still included in all numerical summaries below.

# Filter dataset to remove zero amounts (required for log scale visualization)
plot_df = df.loc[df["Amount"] > 0, ["Amount", "Class"]].copy()

# Replace numeric class labels with readable labels for plotting
plot_df["Class"] = plot_df["Class"].map(CLASS_LABELS)

# Create ECDF plot with class-based coloring
fig = px.ecdf(
    plot_df,
    x="Amount",      
    color="Class",   
    log_x=True,      
    marginal="box",  
    title="Amount distribution by class — ECDF on log scale",
)

fig.update_layout(
    xaxis_title="Amount (log scale)",
    yaxis_title="Cumulative share within class",
    legend_title="Class",
)

fig.show()

# Compute summary statistics for transaction amounts by class
amount_summary = pd.DataFrame(
    {
        "median": [
            df.loc[df["Class"] == 0, "Amount"].median(),
            df.loc[df["Class"] == 1, "Amount"].median(),
        ],
        "p90": [
            df.loc[df["Class"] == 0, "Amount"].quantile(0.90),
            df.loc[df["Class"] == 1, "Amount"].quantile(0.90),
        ],
        "p99": [
            df.loc[df["Class"] == 0, "Amount"].quantile(0.99),
            df.loc[df["Class"] == 1, "Amount"].quantile(0.99),
        ],
    },
    index=["Legitimate", "Fraud"],
)

display(amount_summary)

### Answer from the `Amount` Distribution

`Amount` is one of the few directly interpretable variables in the dataset, so we analyze it separately.

The ECDF is more useful than a simple histogram in this case because the two classes have very different sizes. A standard histogram would be dominated by legitimate transactions, while the ECDF compares the cumulative distribution within each class and makes the two shapes easier to read. We also use a logarithmic x-axis because transaction amounts are highly right-skewed.

The results show that fraud transactions are not simply always larger or always smaller than legitimate ones. The median amount is lower for frauds (**9.25**) than for legitimate transactions (**22.00**), meaning that many fraudulent transactions involve small amounts. However, the upper tail is also relevant: the 90th and 99th percentiles are higher for fraud transactions than for legitimate ones. This suggests that fraudulent transactions include both many small transactions and some relatively large ones.

This step does not prove that `Amount` alone can detect fraud. Instead, it shows that `Amount` contains useful information but has a skewed distribution and extreme values. For this reason, it should be scaled carefully before training models, especially models such as Logistic Regression that can be affected by feature magnitude and regularization.



In [ ]:
# Compute the correlation matrix for all numeric features in the dataset
corr = df.corr(numeric_only=True)

# Visualize the correlation matrix using an interactive heatmap
fig = px.imshow(
    corr,
    color_continuous_scale="RdBu_r",  
    zmin=-1,                          
    zmax=1,                           
    aspect="auto",                    
    title="Correlation heatmap including the target variable",
)

fig.update_xaxes(side="bottom")

fig.update_layout(
    width=900,
    height=750,
    coloraxis_colorbar_title="corr"
)

fig.show()

# Extract and rank correlations with the target variable (Class)
if "Class" in corr.columns:
    top_corr = (
        corr["Class"]              
        .drop("Class")             
        .abs()                     
        .sort_values(ascending=False)
        .head(10)                  
    )

    print("Top absolute correlations with Class (linear):")
    display(top_corr.to_frame(name="|corr(Class)|"))

### Answer from the Correlation Check

The correlation heatmap gives us a first view of the linear relationships between the numerical variables, including the target variable `Class`.

The anonymized variables `V1` to `V28` show very low correlation with each other, which is consistent with the fact that they are transformed components. More importantly for this project, the ranking of absolute correlations with `Class` shows that some variables are more linearly associated with the target than others. In particular, **V17**, **V14**, **V12**, and **V10** appear among the strongest linear signals.

This result must be interpreted carefully. Correlation does not imply causation, and it only captures linear relationships. Moreover, since most variables are anonymized, we cannot assign them a direct business meaning or explain them in terms of real transaction characteristics.

The useful conclusion is therefore limited but still important: some anonymized components seem to contain information related to fraud detection. This supports the idea that the models may be able to learn useful patterns from the features, even if we cannot directly interpret most of them. It also motivates comparing a simple linear model, such as Logistic Regression, with more flexible ensemble models, such as Random Forest and XGBoost.



## 5. Data Preparation

After the exploratory checks, we prepare the data for modelling.

The preparation choices are based on what we observed in the previous sections:

1. `Class` is the target variable, while all the other columns are used as input features;
2. the class distribution is highly imbalanced, so the train/test split must be stratified to preserve the fraud ratio in both sets;
3. the test set must remain untouched until the final evaluation, so that it can represent unseen data;
4. most variables are anonymized transformed components, while `Time` and `Amount` are directly interpretable variables with a different scale;
5. `Time` and `Amount` are therefore scaled before modelling;
6. the scaler is fitted only on the training set and then applied to the test set, in order to avoid data leakage.

This step is important because the quality of the data preparation affects the reliability of the models. In particular, keeping the test set separated and avoiding leakage allows us to evaluate the models in a more realistic way.


In [ ]:
# Count fully duplicated rows in the dataset
duplicate_rows = int(df.duplicated().sum())

# Create a mask identifying transactions with zero amount
zero_amount_mask = df["Amount"] == 0

# Build a summary comparing zero-amount vs non-zero-amount transactions
amount_zero_summary = (
    df.assign(_zero=zero_amount_mask)  # temporary flag column for grouping
    .groupby("_zero")["Class"]         # group by zero vs non-zero amount
    .agg(
        n="count",        
        frauds="sum",     
        fraud_rate="mean" 
    )
    .rename(index={False: "Amount > 0", True: "Amount == 0"})  
)

print("\nFully duplicated rows:", duplicate_rows)

print("\nSummary by Amount zero vs positive:")
print(amount_zero_summary.to_string())

### Data-Quality Notes

There are **1,081 fully duplicated rows** in the dataset. We count them because duplicates can influence model training and evaluation, especially in classification tasks. However, in this version of the project, we do not remove them. The reason is methodological: we want to keep the dataset as provided and compare all models on the same input data, without changing the benchmark during the notebook.

We also check zero-amount transactions because `Amount` is one of the few directly interpretable variables. The results show that transactions with `Amount == 0` have a higher fraud rate than transactions with a positive amount. For this reason, we do not treat them as automatic errors and we do not remove them. They may contain useful information for fraud detection, so they are kept in the dataset.


In [ ]:
# Define feature columns by excluding the target variable
FEATURE_COLS = [col for col in df.columns if col != "Class"]

# Split dataset into features (X) and target (y)
X = df[FEATURE_COLS].copy()
y = df["Class"].copy()

# Split data into training and testing sets
# stratify=y ensures the same class distribution in both sets (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,          # 20% of data used for testing
    stratify=y,              # preserve class imbalance ratio
    random_state=RANDOM_STATE # ensure reproducibility
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

print("\nTrain class counts:\n", y_train.value_counts().sort_index())

print("\nTest class counts:\n", y_test.value_counts().sort_index())

print("\nTrain fraud rate (mean of Class):", float(y_train.mean()))

print("Test fraud rate (mean of Class):", float(y_test.mean()))

### Why We Use a Stratified Split

The dataset is highly imbalanced, because fraudulent transactions represent only a very small fraction of all observations. For this reason, we do not use a simple random split without control over the class distribution.

Instead, we use a stratified train/test split. Stratification means that the split preserves approximately the same proportion of legitimate and fraudulent transactions in both the training set and the test set. This is important because, with such a rare positive class, a normal random split could accidentally create a test set with too few or too many fraud cases, making the final evaluation less reliable.

The results confirm that the fraud rate is almost identical in the training set and in the test set. This means that both sets represent the original imbalance of the dataset.

The test set is kept completely separate from the modelling process. It is not used to train the models, choose hyperparameters, or fit preprocessing transformations. In particular, scaling is performed after the split: the scaler is fitted only on the training data and then applied to the test data. This avoids data leakage and gives a more realistic estimate of how the model would perform on unseen transactions.


In [ ]:
# RobustScaler is used to scale features in a way that is resistant to outliers.
# It is fitted ONLY on the training set to avoid data leakage from test data.

# Initialize the scaler
scaler = RobustScaler()

# Create copies of the training and test sets to avoid modifying original data
X_train_t = X_train.copy()
X_test_t = X_test.copy()

scale_cols = ["Time", "Amount"]

# Fit the scaler on training data and transform training set
X_train_t[scale_cols] = scaler.fit_transform(X_train[scale_cols])

# Apply the same transformation to the test set (no refitting)
X_test_t[scale_cols] = scaler.transform(X_test[scale_cols])

print("Train shape:", X_train_t.shape, "| Test shape:", X_test_t.shape)

# Check scaled range for Time feature (training set)
print(
    "Scaled Time   -> min:", round(X_train_t["Time"].min(), 2),
    " max:", round(X_train_t["Time"].max(), 2),
)

# Check scaled range for Amount feature (training set)
print(
    "Scaled Amount -> min:", round(X_train_t["Amount"].min(), 2),
    " max:", round(X_train_t["Amount"].max(), 2),
)

### Why We Scale Only `Time` and `Amount`

The variables `V1` to `V28` are anonymized PCA-transformed components. This means that they have already been transformed from the original transaction features, mainly for confidentiality reasons. `Time` and `Amount`, instead, are not PCA-transformed and still keep their original scale.

For this reason, we scale only `Time` and `Amount`. We use `RobustScaler` because it scales features using the median and the interquartile range, making it more resistant to extreme values than standard scaling. This choice is especially useful for `Amount`, which is strongly right-skewed and contains very large values.

The scaler is fitted only on the training set and then applied to the test set. This avoids data leakage, because no information from the test distribution is used during preprocessing.


## 6. Modelling Strategy

At this point, we have enough information to decide how to approach the modelling phase.

We compare three supervised classification model families with different levels of complexity:

1. **Logistic Regression**: a simple and interpretable linear baseline. It estimates the probability that a transaction belongs to the fraud class and helps us understand how far a transparent model can go.

2. **Random Forest**: an ensemble model based on many decision trees. Each tree gives a prediction, and the final output is obtained by aggregating their decisions. This model can capture non-linear relationships and interactions between variables, making it a strong candidate for tabular data.

3. **XGBoost**: a boosted-tree model. Unlike Random Forest, which builds trees mostly independently, XGBoost builds trees sequentially, with each new tree trying to correct the errors made by the previous ones. This makes it a more advanced and usually very competitive model for structured datasets.

The goal is not only to find the highest number in a table. We want to understand whether increasing model complexity improves fraud detection, and whether hyperparameter tuning actually improves the results compared to the default version of each model.



## Evaluation Strategy

Before training and comparing the models, we define a common evaluation function. This allows us to evaluate all classifiers using the same metrics and the same decision threshold.

Each model will produce a probability score for the fraud class. The evaluation function converts these probabilities into class predictions using a threshold of **0.5**. This threshold is a standard starting point, but in an imbalanced fraud detection problem it may not be the optimal business threshold. For this reason, we also evaluate ranking metrics such as ROC-AUC and PR-AUC, which use the predicted probabilities directly.

The function computes:

* **ROC-AUC**, to measure the general ability of the model to separate legitimate and fraudulent transactions;
* **PR-AUC**, which is especially important because the positive class is rare;
* **precision**, to understand how many predicted frauds are actually frauds;
* **recall**, to understand how many real frauds are detected;
* **F1-score**, to summarize the balance between precision and recall at the selected threshold.

We also store each result in a shared list, so that all models can be compared later in a final leaderboard.

Finally, we define a stratified cross-validation strategy. Stratification is important because the dataset is highly imbalanced, so each fold should preserve approximately the same proportion of legitimate and fraudulent transactions. We use two folds to keep the search computationally manageable on a large dataset. With more time and resources, this could be increased to three or five folds.



In [ ]:
MODEL_RESULTS = []


def evaluate_model(name, y_true, proba, threshold=0.5):
    """Evaluate a probabilistic binary classifier on the test set."""
    y_pred = (proba >= threshold).astype(int) #convert probabilities to class predictions
    row = {
        "name": name,
        "roc_auc": roc_auc_score(y_true, proba), #area under the ROC curve
        "pr_auc": average_precision_score(y_true, proba), #area under the PR curve
        "precision@0.5": precision_score(y_true, y_pred, zero_division=0), 
        "recall@0.5": recall_score(y_true, y_pred, zero_division=0), 
        "f1@0.5": f1_score(y_true, y_pred, zero_division=0),
    }
    print(pd.Series(row).to_string())
    MODEL_RESULTS.append(row) 
    return row


# Two folds keep the search light on a large dataset.
# With more time/resources, this could be increased to 3 or 5 folds.
CV = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)


At this point, we have a consistent evaluation setup. This allows us to train different models and compare them using the same metrics. The test set remains reserved for final evaluation, while stratified cross-validation will be used during hyperparameter tuning.

## 6.1 Logistic Regression

We start with Logistic Regression because it is a good baseline model for binary classification.

Logistic Regression is simple, fast, and interpretable. It estimates the probability that each transaction belongs to the fraud class and then converts this probability into a class prediction using a decision threshold.

Before running it, the expectation is clear: since it learns a linear decision rule, it may not capture all fraud patterns. Still, it is useful because it gives us a transparent reference point. If a simple model already performs well, more complex models must justify their additional complexity.


In [ ]:
lr_default = LogisticRegression( 
    max_iter=5000, #max n of iterations
    random_state=RANDOM_STATE, #ensure reproducibility
    solver="saga", 
)
lr_default.fit(X_train_t, y_train)

p_lr_default = lr_default.predict_proba(X_test_t)[:, 1] #predict the probability of fraud for each transaction
print("=== LogisticRegression (default) — test ===")
row_lr_default = evaluate_model("LR default", y_test, p_lr_default)


### Logistic Regression Baseline Result

The Logistic Regression baseline reaches **ROC-AUC = 0.961** and **PR-AUC = 0.744** on the test set.

ROC-AUC is high, meaning that the model is generally able to rank fraudulent transactions above legitimate ones. However, in this project PR-AUC is more informative, because the fraud class is extremely rare and we care specifically about the precision-recall trade-off for the positive class.

At the default threshold of **0.5**, the model obtains **precision = 0.829** and **recall = 0.643**. This means that when the model flags a transaction as fraud, it is often correct, but it still misses a relevant share of real fraud cases.

This gives us a first useful baseline: a simple linear classifier can detect meaningful structure in the data, but its recall at the standard threshold leaves room for improvement.


After testing the baseline Logistic Regression model, we try to improve it through hyperparameter tuning.

We tune two parameters:

* `C`: controls the regularization strength. A smaller value means stronger regularization and a simpler model, while a larger value allows the model to fit the training data more closely.
* `class_weight`: controls whether the model treats the two classes equally or gives more importance to the minority class. This is relevant because fraud cases are much rarer than legitimate transactions.

Since the search space is small, we use Grid Search. The model tests all parameter combinations using stratified cross-validation and selects the one with the best average precision score. We use average precision because it is aligned with PR-AUC and is more suitable than accuracy for this imbalanced problem.


In [ ]:
param_lr = {
    "C": [0.1, 1.0, 10.0], #controls the regularization strength. A smaller value makes the model simpler and more regularized, while a larger value allows the model to fit the training data more closely.
    "class_weight": [None, "balanced"], #controls whether the model should treat both classes equally or give more importance to minority class.
}

g_lr = GridSearchCV(
    LogisticRegression(max_iter=10000, random_state=RANDOM_STATE, solver="saga"), 
    param_grid=param_lr, #grid search to find the best hyperparameters
    scoring="average_precision", #we use average precision because fraud is rare
    cv=CV, 
    n_jobs=-1, #all available cores
    refit=True, 
)
g_lr.fit(X_train_t, y_train) 

print("Best params:", g_lr.best_params_) 
print("Best CV average_precision:", round(g_lr.best_score_, 6)) #best average precision score

p_lr_tuned = g_lr.predict_proba(X_test_t)[:, 1] 
print("\n=== LogisticRegression (tuned) — test ===") 
row_lr_tuned = evaluate_model("LR tuned", y_test, p_lr_tuned)


### Logistic Regression Tuning Result

The grid search selects **`C = 0.1`** and **`class_weight = None`**.

Compared with the baseline Logistic Regression, the tuned model slightly improves ROC-AUC, but PR-AUC decreases a little. Precision, recall, and F1-score at the 0.5 threshold remain unchanged.

This suggests that tuning these hyperparameters does not substantially improve the Logistic Regression model. The model is useful as a strong and interpretable baseline, but we now need to compare it with more flexible models to see whether they can better capture non-linear patterns and improve fraud detection.



### Logistic Regression: Baseline vs Tuned Comparison

Before moving to a more complex model, we compare the baseline and tuned versions of Logistic Regression side by side. This helps us check whether hyperparameter tuning produced a meaningful improvement or whether the model has already reached its practical limit with this linear approach.

In [ ]:
# Compare Logistic Regression baseline and tuned results
lr_comparison = pd.DataFrame([row_lr_default, row_lr_tuned])

display(lr_comparison)

# Convert results to long format for plotting
lr_plot = lr_comparison.melt(
    id_vars="name",
    value_vars=["roc_auc", "pr_auc", "precision@0.5", "recall@0.5", "f1@0.5"],
    var_name="Metric",
    value_name="Score",
)

# Interactive grouped bar chart
fig = px.bar(
    lr_plot,
    x="Metric",
    y="Score",
    color="name",
    barmode="group",
    text=lr_plot["Score"].map(lambda x: f"{x:.3f}"),
    title="Logistic Regression: baseline vs tuned",
)

fig.update_traces(
    textposition="outside",
    hovertemplate="Metric: %{x}<br>Score: %{y:.4f}<extra></extra>",
)

fig.update_layout(
    xaxis_title="Metric",
    yaxis_title="Score",
    yaxis_range=[0, 1],
    legend_title="Model",
)

fig.show()

The comparison shows that tuning does not produce a clear improvement over the baseline Logistic Regression. ROC-AUC slightly increases, but PR-AUC slightly decreases, while precision, recall, and F1-score at the 0.5 threshold remain unchanged.

For this reason, we keep Logistic Regression as a useful and interpretable baseline, but we now move to more flexible models to test whether non-linear methods can improve fraud detection.

## 6.2 Random Forest

The next model is Random Forest. This is a natural second step because it remains within classical machine learning models, but it is more flexible than Logistic Regression.

While Logistic Regression learns a single linear decision rule, Random Forest combines many decision trees and can capture more complex, non-linear relationships between the transaction features and the target variable.

A single decision tree can be unstable and sensitive to noise, because small changes in the data may produce a different tree structure. Random Forest reduces this problem by training many trees and aggregating their predictions. For classification, the final prediction is based on the combined output of the trees, while the predicted probability is obtained from the average class probabilities across the forest.

In this first version, we do not perform hyperparameter tuning yet. Instead, we manually choose a reasonable starting configuration, including the number of trees, the maximum depth of each tree, and the minimum number of samples required in each leaf. This gives us an initial Random Forest result that we can later compare with the tuned version.


In [ ]:
rf_default = RandomForestClassifier(
    n_estimators=200,#builds 200 decision trees
    max_depth=12, #each tree can grow up to depth 12. This limits complexity and helps avoid overfitting
    min_samples_leaf=2, 
    random_state=RANDOM_STATE, #fixes randomness so results are reproducible.
    n_jobs=-1,
)
rf_default.fit(X_train_t, y_train) 

p_rf_default = rf_default.predict_proba(X_test_t)[:, 1] #predict the probability of fraud for each trasaction 
print("=== RandomForest (default-ish) — test ===")
row_rf_default = evaluate_model("RF default", y_test, p_rf_default) #evaluate the model 


### Random Forest Baseline Result

The initial Random Forest clearly improves over Logistic Regression. PR-AUC increases to **0.868**, which is a strong improvement for this imbalanced classification problem.

At the threshold of **0.5**, the model reaches **precision = 0.942** and **recall = 0.827**. This means that the model detects a larger share of real frauds while also keeping a low number of false alarms among the transactions predicted as fraud.

This result suggests that a more flexible model can capture useful patterns that the linear Logistic Regression model may miss. The improvement is not due to changing the evaluation metric, because the same metrics and the same test set are used. The gain comes from using a model family that can learn more complex relationships between the features.



After the initial Random Forest model, we try to improve its performance by tuning some of its main hyperparameters.

We define a set of possible values for parameters that are usually important in Random Forest models:

* `n_estimators`: the number of trees in the forest. More trees can make the model more stable, but they also increase training time.
* `max_depth`: the maximum depth of each tree. Deeper trees can capture more complex patterns, but they can also increase the risk of overfitting.
* `min_samples_leaf`: the minimum number of samples required in a final leaf node. Higher values make the trees more conservative and can reduce overfitting.
* `class_weight`: controls whether the model gives more importance to the minority class. This is relevant because fraudulent transactions are much rarer than legitimate ones.

Since the full search space contains many possible combinations, we use `RandomizedSearchCV`. Instead of testing every possible combination, it evaluates only a limited number of randomly selected configurations. In this project, we test 8 combinations to keep the computation manageable.

The best combination is selected according to average precision, which is aligned with PR-AUC and is more suitable than accuracy for an imbalanced fraud detection problem.


In [ ]:
param_rf = {
    "n_estimators": [200, 400], 
    "max_depth": [8, 12, 16, None], #none means that the tree can grow as deep as needed to reduce impurity
    "min_samples_leaf": [1, 2, 4], #highest values make the model more conservative and reduce overfitting
    "class_weight": [None, "balanced"],
}

r_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_rf,
    n_iter=8, #randomly selects 8 combinations from the grid
    scoring="average_precision", #we use average precision because fraud is rare
    cv=CV,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)
r_rf.fit(X_train_t, y_train)

print("Best params:", r_rf.best_params_)
print("Best CV average_precision:", round(r_rf.best_score_, 6))

p_rf_tuned = r_rf.predict_proba(X_test_t)[:, 1] #the best combination is used to predict the probability of fraud for each transaction
print("\n=== RandomForest (tuned) — test ===")
row_rf_tuned = evaluate_model("RF tuned", y_test, p_rf_tuned)


### Random Forest Tuning Result

The randomized search selects **200 trees**, **max_depth = 16**, **min_samples_leaf = 1**, and **no class weighting**.

The tuned Random Forest slightly improves PR-AUC, from **0.868** to **0.872**. ROC-AUC also increases, from **0.974** to **0.979**. However, precision, recall, and F1-score at the default threshold of **0.5** remain unchanged.

This means that tuning improves the ranking quality of the model, because the probability scores become slightly better according to ROC-AUC and PR-AUC. However, this improvement is not enough to change the final class predictions at the 0.5 threshold.

Overall, hyperparameter tuning helps, but only marginally. Most of the improvement came from moving from a linear model to a tree-based ensemble. The tuned Random Forest remains a strong candidate, but we now continue with XGBoost to test whether a boosted-tree approach can further improve fraud detection.


### Random Forest: Baseline vs Tuned Curves

The threshold-based metrics at 0.5 are almost identical for the two Random Forest versions, so a curve-based comparison is more informative here.

ROC and Precision–Recall curves allow us to compare the models across all possible thresholds, not only at 0.5. This is especially useful in fraud detection, where the positive class is rare and the business threshold may change depending on how conservative we want to be.

In particular, the Precision–Recall curve is more relevant for this project because it focuses on performance on the minority fraud class.

In [ ]:
# --- ROC data ---
fpr_rf_default, tpr_rf_default, _ = roc_curve(y_test, p_rf_default)
fpr_rf_tuned, tpr_rf_tuned, _ = roc_curve(y_test, p_rf_tuned)

roc_auc_default = auc(fpr_rf_default, tpr_rf_default)
roc_auc_tuned = auc(fpr_rf_tuned, tpr_rf_tuned)

fig_roc = go.Figure()

fig_roc.add_trace(go.Scatter(
    x=fpr_rf_default,
    y=tpr_rf_default,
    mode="lines",
    name=f"RF baseline (AUC = {roc_auc_default:.3f})"
))

fig_roc.add_trace(go.Scatter(
    x=fpr_rf_tuned,
    y=tpr_rf_tuned,
    mode="lines",
    name=f"RF tuned (AUC = {roc_auc_tuned:.3f})"
))

fig_roc.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Random classifier",
    line=dict(dash="dash")
))

fig_roc.update_layout(
    title="Random Forest: ROC curve comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate (Recall)",
    width=800,
    height=600,
)

fig_roc.show()


# --- Precision-Recall data ---
precision_rf_default, recall_rf_default, _ = precision_recall_curve(y_test, p_rf_default)
precision_rf_tuned, recall_rf_tuned, _ = precision_recall_curve(y_test, p_rf_tuned)

pr_auc_default = average_precision_score(y_test, p_rf_default)
pr_auc_tuned = average_precision_score(y_test, p_rf_tuned)

baseline_positive_rate = y_test.mean()

fig_pr = go.Figure()

fig_pr.add_trace(go.Scatter(
    x=recall_rf_default,
    y=precision_rf_default,
    mode="lines",
    name=f"RF baseline (AP = {pr_auc_default:.3f})"
))

fig_pr.add_trace(go.Scatter(
    x=recall_rf_tuned,
    y=precision_rf_tuned,
    mode="lines",
    name=f"RF tuned (AP = {pr_auc_tuned:.3f})"
))

fig_pr.add_trace(go.Scatter(
    x=[0, 1],
    y=[baseline_positive_rate, baseline_positive_rate],
    mode="lines",
    name="No-skill baseline",
    line=dict(dash="dash")
))

fig_pr.update_layout(
    title="Random Forest: Precision–Recall curve comparison",
    xaxis_title="Recall",
    yaxis_title="Precision",
    width=800,
    height=600,
)

fig_pr.show()

# What the Random Forest Curves Show

The ROC and Precision–Recall curves confirm the numerical comparison between the initial and tuned Random Forest models.

The ROC curves are very close, with the tuned model showing only a small improvement in ROC-AUC. This means that both models are already very good at ranking fraudulent transactions above legitimate ones.

The Precision–Recall curve is more important for this project because fraud is extremely rare. Here again, the tuned model shows a slight advantage, with AP increasing from **0.868** to **0.872**. However, the two curves remain very similar, so the improvement is limited.

Overall, hyperparameter tuning slightly improves the quality of the probability ranking, but it does not radically change the model. The main improvement came from moving from Logistic Regression to Random Forest, while tuning adds only a small additional gain.

## 6.3 XGBoost

The last model family we test is XGBoost. Like Random Forest, it is based on decision trees, but it uses a different learning strategy.

Random Forest trains many trees mostly independently and aggregates their predictions. XGBoost, instead, builds trees sequentially: each new tree tries to correct the errors left by the previous ones. This boosting logic can be useful in difficult classification problems, because the model progressively focuses on observations that are harder to classify.

Since the dataset is highly imbalanced, we also compute `scale_pos_weight` from the training data. This parameter gives more weight to the minority class during training, helping the model pay more attention to fraudulent transactions. This is important because fraud cases are much rarer than legitimate transactions.


In [ ]:
pos = int((y_train == 1).sum()) #n of fraud cases
neg = int((y_train == 0).sum()) #n of legitimate cases
scale_pos_weight = max(neg / max(pos, 1), 1.0) #to balance the classes. legitimate>fraud

xgb_default = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.08,#controls how much each tree contributes to the final predictions
    subsample=0.8,
    colsample_bytree=0.8,#reduce overfitting
    reg_lambda=1.0, #controls the regularizaion strenght
    random_state=RANDOM_STATE,
    eval_metric="logloss", 
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1, 
    tree_method="hist", #use histogram-based tree method
)
xgb_default.fit(X_train_t, y_train)

p_xgb_default = xgb_default.predict_proba(X_test_t)[:, 1]
print("=== XGBoost (default-ish) — test ===")
row_xgb_default = evaluate_model("XGB default", y_test, p_xgb_default)


### XGBoost Baseline Result

The initial XGBoost model reaches **ROC-AUC = 0.980** and **PR-AUC = 0.868**, which is close to the Random Forest results.

At the default threshold of **0.5**, XGBoost obtains **recall = 0.857** and **precision = 0.778**. Compared with Random Forest, it detects a slightly larger share of real frauds, but it also produces more false positives. In practical terms, this means that XGBoost is more aggressive in flagging transactions as fraud.

This is an important result because it shows why looking at only one metric is risky. A model with higher recall may be preferable if the business priority is to miss as few frauds as possible. However, lower precision means more legitimate transactions are incorrectly flagged, which can create customer friction and operational costs.

Therefore, the preferred model depends not only on technical performance, but also on the business cost of false positives and false negatives.



After the initial XGBoost model, we try to improve its performance through hyperparameter tuning and then analyze how the decision threshold affects the final predictions.

For XGBoost, the search space is larger than for the previous models, because the algorithm has several parameters that control tree complexity, learning speed, sampling, regularization, and class imbalance. For this reason, we use `RandomizedSearchCV` instead of testing every possible combination. This allows us to explore many possible configurations while keeping the computation manageable.

The model is tuned using average precision as the scoring metric, because it is aligned with PR-AUC and is more appropriate than accuracy for this highly imbalanced fraud detection problem. In particular, we also tune `scale_pos_weight`, which controls how much importance is given to the minority fraud class during training.

After selecting the best XGBoost configuration, we evaluate it at the standard threshold of **0.5**. However, in fraud detection, this default threshold is not necessarily the best operational choice. A lower threshold usually increases recall and reduces missed frauds, but it also creates more false positives. A higher threshold usually increases precision, but it may miss more real frauds.

For this reason, we also test multiple thresholds after training the model. This does not change the model itself; it only changes how predicted fraud probabilities are converted into final class predictions. The goal is to understand the trade-off between precision, recall, false positives, and false negatives, and to select a threshold that better reflects the business priority of reducing missed frauds.


In [ ]:


param_xgb = {
    # Number of boosting rounds: keep around the best area, without making it too heavy
    "n_estimators": [500, 600, 700, 800],

    # Tree complexity: medium-depth trees usually work well for tabular fraud data
    "max_depth": [4, 5, 6],

    # Learning speed: smaller values are safer, but require enough estimators
    "learning_rate": [0.04, 0.05, 0.06, 0.07],

    # Row sampling: helps reduce overfitting
    "subsample": [0.7, 0.75, 0.8, 0.85],

    # Feature sampling
    "colsample_bytree": [0.9, 0.95, 1.0],

    # L2 regularization
    "reg_lambda": [6.0, 8.0, 10.0, 12.0],

    # L1 regularization: small values can help control complexity
    "reg_alpha": [0.05, 0.1, 0.2],

    # Controls how conservative the tree splits are
    "min_child_weight": [3, 4, 5, 6],

    # Minimum loss reduction required to split a node
    "gamma": [0.05, 0.1, 0.15],

    # Useful for imbalanced logistic classification
    "max_delta_step": [1, 2, 3],

    # Explore different weights for the minority fraud class
    "scale_pos_weight": [
        scale_pos_weight * 0.4,
        scale_pos_weight * 0.5,
        scale_pos_weight * 0.6,
        scale_pos_weight * 0.7,
        scale_pos_weight * 0.8,
    ],
}

r_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=RANDOM_STATE,
        objective="binary:logistic",
        eval_metric="aucpr",
        n_jobs=-1,
        tree_method="hist",
    ),
    param_distributions=param_xgb,
    n_iter=50,                       # randomly selects 50 combinations from the grid
    scoring="average_precision",     # aligned with PR-AUC
    cv=CV,
    random_state=RANDOM_STATE + 2,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

r_xgb.fit(X_train_t, y_train)

print("Best params:", r_xgb.best_params_)
print("Best CV average_precision:", round(r_xgb.best_score_, 6))

# Probability of fraud for each transaction
p_xgb = r_xgb.predict_proba(X_test_t)[:, 1]

print("\n=== XGBoost (tuned) — test at threshold 0.5 ===")
row_xgb = evaluate_model("XGB tuned", y_test, p_xgb)


# ------------------------------------------------------------
# Threshold policy analysis
# ------------------------------------------------------------

def metrics_at_threshold(y_true, proba, threshold):
    """Compute classification metrics at a specific probability threshold."""
    y_pred = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "threshold": threshold,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "predicted_frauds": int(y_pred.sum()),
    }


# Test different thresholds between 0.05 and 0.95
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)

threshold_results = pd.DataFrame(
    [metrics_at_threshold(y_test, p_xgb, t) for t in thresholds]
)

print("\n=== XGBoost tuned — threshold analysis ===")
display(threshold_results)


# Plot precision, recall and F1 across thresholds
threshold_plot = threshold_results.melt(
    id_vars="threshold",
    value_vars=["precision", "recall", "f1"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    threshold_plot,
    x="threshold",
    y="score",
    color="metric",
    markers=True,
    title="XGBoost tuned — precision, recall and F1 across thresholds",
)

fig.update_layout(
    xaxis_title="Decision threshold",
    yaxis_title="Score",
    yaxis_range=[0, 1],
    legend_title="Metric",
)

fig.show()


# Business-oriented threshold policy:
# We prefer reducing missed frauds, so we set a minimum recall target.
# Among thresholds that reach this recall, we select the one with the highest precision.

TARGET_RECALL = 0.85

candidate_thresholds = threshold_results[
    threshold_results["recall"] >= TARGET_RECALL
].copy()

if len(candidate_thresholds) > 0:
    selected_threshold_row = candidate_thresholds.sort_values(
        ["precision", "threshold"],
        ascending=[False, False]
    ).iloc[0]
else:
    selected_threshold_row = threshold_results.sort_values(
        "recall",
        ascending=False
    ).iloc[0]

selected_threshold = float(selected_threshold_row["threshold"])

print("\n=== Selected threshold policy ===")
print(f"Target recall: {TARGET_RECALL}")
display(selected_threshold_row.to_frame().T)

### XGBoost Tuning Result

The tuned XGBoost model reaches **ROC-AUC = 0.981** and **PR-AUC = 0.877**, which is the highest PR-AUC obtained so far in the notebook.

Compared with the initial XGBoost model, the tuned version improves PR-AUC from **0.868** to **0.877**. At the default threshold of **0.5**, precision increases from **0.778** to **0.828**, while recall decreases slightly from **0.857** to **0.837**. This means that the tuned model produces fewer false positives, but it also misses slightly more frauds at this specific threshold.

However, PR-AUC evaluates the ranking quality of the probability scores across thresholds. Since the tuned XGBoost model has the best PR-AUC, we can use it as the strongest ranking model and then choose a decision threshold that better matches the business objective.



### XGBoost Threshold Policy Result

The threshold analysis shows how the tuned XGBoost model behaves when we change the decision threshold used to convert fraud probabilities into final predictions.

At the standard threshold of **0.5**, the model obtains **precision = 0.828**, **recall = 0.837**, and **F1-score = 0.832**. This corresponds to **82 true frauds detected**, **17 false positives**, and **16 missed frauds**.

Since this is a fraud detection problem, we prefer a conservative policy that reduces false negatives, even if this creates some additional false positives. For this reason, we set a minimum target recall of **0.85** and then choose the threshold with the highest precision among the thresholds that satisfy this condition.

The selected threshold is **0.40**. At this threshold, the model reaches **recall = 0.857** and **precision = 0.785**. In practical terms, it detects **84 frauds** and misses **14 frauds**, compared with **82 detected frauds** and **16 missed frauds** at the default threshold of 0.5. The cost of this improvement is an increase in false positives from **17** to **23**.

This trade-off is acceptable for our business interpretation of the problem. Missing a real fraud is usually more costly than temporarily flagging a legitimate transaction for review. Therefore, the threshold of **0.40** is a more conservative operating policy than the default threshold, because it catches more frauds while keeping the number of false alarms still relatively controlled.


## 7. Model Comparison

Now we compare all trained models on the same test set.

The table is sorted by **PR-AUC** because, after inspecting the class distribution, we know that this is a rare positive-class problem. In this context, PR-AUC is more informative than plain accuracy because it focuses more directly on the precision-recall trade-off for the fraud class.

In [ ]:
summary = pd.DataFrame(MODEL_RESULTS) #create a dataframe with the results
summary = summary.sort_values("pr_auc", ascending=False).reset_index(drop=True) #sort it by PR-AUC

display(summary) 
print("\nSorted by pr_auc (higher is better for finding fraud under imbalance).") 


### Reading the Leaderboard

The best PR-AUC is obtained by **XGB tuned**, with **PR-AUC = 0.877**. This means that, overall, the tuned XGBoost model provides the best ranking performance for the rare fraud class.

The tuned Random Forest is slightly lower in PR-AUC, with **PR-AUC = 0.872**, but it has the strongest F1-score at the default threshold of **0.5**. It also has higher precision than XGBoost, meaning that when it flags a transaction as fraud, it is more often correct.

XGBoost tuned, instead, has slightly higher recall than Random Forest tuned at the same threshold. This means that it catches a few more fraud cases, but at the cost of more false positives.

Therefore, the conclusion is not simply that one model is always best. The decision depends on the operational objective:

* **XGB tuned** is the strongest model if the priority is ranking suspicious transactions, because it has the highest PR-AUC;
* **RF tuned** is very strong if the priority is a better default-threshold balance, with higher precision and F1-score;
* the final practical choice should also consider the decision threshold, because the default threshold of 0.5 is not necessarily optimal for fraud detection.

For this reason, we select **XGB tuned** as the strongest ranking model, but we analyze its threshold policy separately in the next section.


## 8. Business Impact and Threshold Policy

In fraud detection, false positives and false negatives do not have the same business cost.

A **false positive** happens when a legitimate transaction is incorrectly flagged as fraud. This can create customer friction, block a valid payment, and generate additional investigation costs. However, the transaction can usually be reviewed or confirmed.

A **false negative** happens when a real fraudulent transaction is classified as legitimate. This is usually more dangerous because it can generate direct financial loss, security risks, and loss of customer trust.

For this reason, in this project we prefer a slightly conservative fraud detection approach. This means that we are willing to accept some additional false positives if this helps reduce the number of missed frauds.

The tuned XGBoost model has the best PR-AUC, so it is the strongest model for ranking transactions by fraud risk. However, the default threshold of **0.5** is not necessarily the best operational threshold. The threshold determines how predicted fraud probabilities are converted into final fraud/not-fraud decisions.

At the standard threshold of **0.5**, the tuned XGBoost model detects **82 frauds**, produces **17 false positives**, and misses **16 frauds**. Precision is **0.828**, recall is **0.837**, and F1-score is **0.832**.

To reflect the business priority of reducing missed frauds, we test multiple thresholds and select a policy with a minimum recall target of **0.85**. Among the thresholds that satisfy this condition, we choose the one with the highest precision.

The selected threshold is **0.40**. At this threshold, the model detects **84 frauds**, produces **23 false positives**, and misses **14 frauds**. Recall increases to **0.857**, while precision decreases to **0.785**.

This trade-off is acceptable for our interpretation of the business problem. Compared with the default threshold, the model catches **2 additional frauds** and misses **2 fewer frauds**, while creating **6 additional false positives**. Since missing a real fraud is usually more costly than temporarily flagging a legitimate transaction, the threshold of **0.40** is a more conservative and business-oriented operating policy.


## 9. Final Conclusion

This notebook follows a complete supervised learning workflow for a credit card fraud detection problem.

We began from the dataset, not from the final model. First, we checked that the data was loaded correctly, verified the target column, inspected the class labels, counted missing values, and measured the imbalance. Only after seeing that fraud represents about **0.173%** of transactions did we conclude that accuracy alone was not enough.

The exploratory analysis showed that `Amount` is strongly skewed and that some anonymized PCA variables have stronger linear association with the target than others. This justified a controlled preprocessing step: separating features and target, using a stratified train/test split, and scaling only `Time` and `Amount` with a scaler fitted on the training set only, in order to avoid data leakage.

The modelling comparison shows a clear pattern. Logistic Regression provides a useful and interpretable baseline, but tree-based ensemble models perform better. Random Forest is very strong at the default threshold of **0.5**, especially in precision and F1-score. XGBoost, after tuning, gives the best PR-AUC and therefore the best ranking performance for the rare fraud class.

The final recommendation is not based on accuracy. The best final candidate is **XGB tuned**, because it obtains the highest PR-AUC and provides the strongest ranking of suspicious transactions. However, from a business perspective, the model should not rely blindly on the default threshold of **0.5**.

For this reason, we also define a threshold policy. Since missing a real fraud is usually more costly than temporarily flagging a legitimate transaction, we choose a more conservative threshold of **0.40** for the tuned XGBoost model. This increases recall and reduces the number of missed frauds, while keeping the number of false positives still controlled.

The final practical recommendation is therefore to use **tuned XGBoost as the ranking model**, combined with a **0.40 decision threshold** when the business priority is to reduce missed frauds.

## 10. Limitations and Possible Improvements

This project is intentionally kept close to the course workflow and to the logic of a readable student notebook. However, there are several possible improvements.

First, the threshold policy could be made more advanced by using an explicit cost matrix for false positives and false negatives. In this notebook, we choose the threshold based on a recall target and a qualitative business interpretation, but in a real company the best threshold should be selected using estimated financial and operational costs.

Second, the project could include confusion matrices for the final selected models to make false positives and false negatives even more visible.

Third, the analysis could test the effect of removing duplicated rows, since the dataset contains fully duplicated observations. We kept them in this version to preserve the original benchmark, but this choice could be tested in a future version.

Other possible improvements include using more cross-validation folds if more computation time is available, analyzing the transactions that are repeatedly misclassified, and comparing the final models with full precision-recall curves.

The main limitation of the dataset is that the variables `V1` to `V28` are anonymized PCA components. This protects privacy, but it makes business interpretation harder. For this reason, the notebook focuses more on predictive performance, methodological correctness, and evaluation under class imbalance than on explaining the meaning of individual features.

